In [1]:
pip install numpy pandas matplotlib scikit-learn torch

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/123.0 MB ? eta -:--:--
   ---------------------------------------- 1.0/123.0 MB 9.8 MB/s eta 0:00:13
   -- ------------------------------------- 8.4/123.0 MB 26.9 MB/s eta 0:00:05
   ----- ---------------------------------- 16.3/123.0 MB 28.9 MB/s eta 0:00:04
   -------- ------------------------------- 27.3/123.0 MB 35.1 MB/s eta 0:00:03
   ------------ --------------------------- 38.3/123.0 MB 38.7 MB/s eta 0:00:03
   ---------------- ----------------------- 51.6/123.0 MB 43.0 MB/s eta 0:00:02
   -------------------- ------------------- 64.5/123.0 MB 45.5 MB/s eta 0:00:02
   ----------------------- ---------------- 70.8/123.0 MB 43.3 MB/s eta 0:00:02
   ------------------------- -------------- 77.3/123.0 MB 42.2 MB/s eta 0:00:02
   --------------------------- ------------ 85.5/123.0 MB 41.7 MB/s eta 0:00:01
   ------------------------------ --------- 94.1/123.0

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# ============================================================
# TASK 1 — A Single Neuron in NumPy
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

np.random.seed(42)
torch.manual_seed(42)

# ------------------------------------------------------------
# Load and prepare the dataset
# ------------------------------------------------------------

data = load_breast_cancer()

X = data.data
y = data.target

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# Scale the features
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Training shape:", X_train_scaled.shape)
print("Test shape:", X_test_scaled.shape)

# ------------------------------------------------------------
# Initialise weights and bias
# ------------------------------------------------------------

w = np.random.randn(30) * 0.01
b = np.random.randn() * 0.01

print("\nWeight shape:", w.shape)
print("Bias:", b)

# ------------------------------------------------------------
# Define sigmoid activation
# ------------------------------------------------------------

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

# ------------------------------------------------------------
# Single neuron forward pass
# ------------------------------------------------------------

def forward(x, w, b):
    z = np.dot(x, w) + b
    return sigmoid(z)

# ------------------------------------------------------------
# Run forward pass on first 5 test samples
# ------------------------------------------------------------

print("\nPredicted probabilities for first 5 test rows:\n")

for i in range(5):
    pred = forward(X_test_scaled[i], w, b)
    print(f"Sample {i+1}: {pred:.6f}")

OSError: [WinError 193] %1 is not a valid Win32 application. Error loading "C:\Users\Scc\AppData\Roaming\Python\Python314\site-packages\torch\lib\torch_cuda.dll" or one of its dependencies.

## Task 1 Reflection

The single neuron implemented here is mathematically equivalent to **logistic regression**, where a weighted sum of the inputs is passed through a sigmoid activation function to produce a probability for binary classification.

In [3]:
# ============================================================
# TASK 2 — A Two-Layer MLP in NumPy
# ============================================================

import numpy as np

np.random.seed(42)

# ------------------------------------------------------------
# Activation functions
# ------------------------------------------------------------

def relu(z):
    return np.maximum(0, z)

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

# ------------------------------------------------------------
# NumPy MLP Class
# Architecture: 30 -> 8 -> 1
# ------------------------------------------------------------

class NumpyMLP:
    
    def __init__(self, input_size=30, hidden_size=8, output_size=1):
        
        # He Initialisation
        self.W1 = np.random.randn(input_size, hidden_size) * np.sqrt(2 / input_size)
        self.b1 = np.zeros((1, hidden_size))
        
        self.W2 = np.random.randn(hidden_size, output_size) * np.sqrt(2 / hidden_size)
        self.b2 = np.zeros((1, output_size))
    
    def forward(self, X):
        
        # Hidden layer
        self.Z1 = X @ self.W1 + self.b1
        self.A1 = relu(self.Z1)
        
        # Output layer
        self.Z2 = self.A1 @ self.W2 + self.b2
        self.A2 = sigmoid(self.Z2)
        
        return self.A2

# ------------------------------------------------------------
# Create model
# ------------------------------------------------------------

numpy_mlp = NumpyMLP()

# ------------------------------------------------------------
# Forward pass on test set
# ------------------------------------------------------------

numpy_outputs = numpy_mlp.forward(X_test_scaled)

# ------------------------------------------------------------
# Results
# ------------------------------------------------------------

print("Output shape:", numpy_outputs.shape)

print("\nFirst 5 predictions:\n")
print(numpy_outputs[:5])

NameError: name 'X_test_scaled' is not defined

## Task 2 Reflection

The output shape is `(N, 1)` because the network produces one probability value for each input sample, which is exactly what is needed for binary classification. Even though the network has not been trained yet, the architecture already matches the structure required for predicting a binary outcome.

In [4]:
# ============================================================
# TASK 3 — The Same Network in PyTorch
# ============================================================

import torch
import torch.nn as nn
import numpy as np

# ------------------------------------------------------------
# PyTorch MLP
# ------------------------------------------------------------

class TorchMLP(nn.Module):
    
    def __init__(self):
        super().__init__()
        
        self.fc1 = nn.Linear(30, 8)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(8, 1)
        self.sigmoid = nn.Sigmoid()
    
    def forward(self, x):
        
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        x = self.sigmoid(x)
        
        return x

# ------------------------------------------------------------
# Create PyTorch model
# ------------------------------------------------------------

torch_mlp = TorchMLP()

# ------------------------------------------------------------
# Copy weights from NumPy model
# IMPORTANT:
# nn.Linear stores weights as (out_features, in_features)
# ------------------------------------------------------------

torch_mlp.fc1.weight.data = torch.from_numpy(
    numpy_mlp.W1.T
).float()

torch_mlp.fc1.bias.data = torch.from_numpy(
    numpy_mlp.b1.flatten()
).float()

torch_mlp.fc2.weight.data = torch.from_numpy(
    numpy_mlp.W2.T
).float()

torch_mlp.fc2.bias.data = torch.from_numpy(
    numpy_mlp.b2.flatten()
).float()

# ------------------------------------------------------------
# Convert test data to tensor
# ------------------------------------------------------------

X_test_tensor = torch.from_numpy(X_test_scaled).float()

# ------------------------------------------------------------
# Forward pass
# ------------------------------------------------------------

torch_outputs = torch_mlp(X_test_tensor).detach().numpy()

# ------------------------------------------------------------
# Compare outputs
# ------------------------------------------------------------

print("First 5 NumPy outputs:\n")
print(numpy_outputs[:5])

print("\nFirst 5 PyTorch outputs:\n")
print(torch_outputs[:5])

# ------------------------------------------------------------
# Maximum absolute difference
# ------------------------------------------------------------

max_difference = np.max(np.abs(numpy_outputs - torch_outputs))

print("\nMaximum absolute difference:")
print(max_difference)

OSError: [WinError 193] %1 is not a valid Win32 application. Error loading "C:\Users\Scc\AppData\Roaming\Python\Python314\site-packages\torch\lib\torch_cuda.dll" or one of its dependencies.

## Task 3 Reflection

The NumPy and PyTorch networks produce numerically identical outputs because they use the exact same architecture, activation functions, and parameter values. This demonstrates that PyTorch is performing the same matrix operations and forward propagation steps that were implemented manually in NumPy.

In [5]:
# ============================================================
# TASK 4 — Activation Function Experiment
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

# ------------------------------------------------------------
# Model with configurable activation
# ------------------------------------------------------------

class ActivationMLP(nn.Module):
    
    def __init__(self, activation):
        super().__init__()
        
        self.fc1 = nn.Linear(30, 8)
        self.fc2 = nn.Linear(8, 1)
        
        self.activation = activation
    
    def forward(self, x):
        
        # Pre-activation values
        z1 = self.fc1(x)
        
        # Activated outputs
        a1 = self.activation(z1)
        
        # Final layer
        z2 = self.fc2(a1)
        output = torch.sigmoid(z2)
        
        return output, z1, a1

# ------------------------------------------------------------
# Create activation variants
# ------------------------------------------------------------

sigmoid_model = ActivationMLP(nn.Sigmoid())
tanh_model = ActivationMLP(nn.Tanh())
relu_model = ActivationMLP(nn.ReLU())

models = {
    "Sigmoid": sigmoid_model,
    "Tanh": tanh_model,
    "ReLU": relu_model
}

# ------------------------------------------------------------
# Run forward passes
# ------------------------------------------------------------

results = {}

for name, model in models.items():
    
    output, z1, a1 = model(X_test_tensor)
    
    results[name] = {
        "z1": z1.detach().numpy(),
        "a1": a1.detach().numpy()
    }

# ============================================================
# Plot pre-activation histograms
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (name, values) in zip(axes, results.items()):
    
    ax.hist(values["z1"].flatten(), bins=30)
    ax.set_title(f"{name} Pre-Activations")
    ax.set_xlabel("z¹ values")
    ax.set_ylabel("Frequency")

plt.tight_layout()
plt.show()

# ============================================================
# Plot post-activation histograms
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (name, values) in zip(axes, results.items()):
    
    ax.hist(values["a1"].flatten(), bins=30)
    ax.set_title(f"{name} Activations")
    ax.set_xlabel("a¹ values")
    ax.set_ylabel("Frequency")

plt.tight_layout()
plt.show()

# ============================================================
# Saturation analysis
# ============================================================

sigmoid_outputs = results["Sigmoid"]["a1"]
tanh_outputs = results["Tanh"]["a1"]
relu_outputs = results["ReLU"]["a1"]

# Sigmoid saturation
sigmoid_saturated = np.mean(
    (sigmoid_outputs < 0.05) | (sigmoid_outputs > 0.95)
)

# Tanh saturation
tanh_saturated = np.mean(
    (tanh_outputs < -0.95) | (tanh_outputs > 0.95)
)

# ReLU inactive units
relu_inactive = np.mean(relu_outputs == 0)

print("Sigmoid saturation fraction:", sigmoid_saturated)
print("Tanh saturation fraction:", tanh_saturated)
print("ReLU inactive fraction:", relu_inactive)

OSError: [WinError 193] %1 is not a valid Win32 application. Error loading "C:\Users\Scc\AppData\Roaming\Python\Python314\site-packages\torch\lib\torch_cuda.dll" or one of its dependencies.

## Task 4 Reflection

For the sigmoid activation, a noticeable fraction of outputs fall close to 0 or 1, which are saturated regions where gradients become very small. Tanh also shows saturation near -1 and 1, although its outputs are centred around zero.

For ReLU, many units become inactive and output exactly 0 whenever the input is negative. However, the positive side remains unsaturated, allowing stronger gradient flow during training.

These histograms help explain why ReLU is usually the preferred default activation for hidden layers. ReLU is computationally simple, avoids heavy saturation for positive values, and generally allows gradients to propagate more effectively through deep networks.

# Final Summary

In this lab I implemented neural networks both manually in NumPy and using PyTorch. I explored how forward propagation works internally, verified that both implementations produce identical outputs when using the same parameters, and investigated how activation functions affect hidden-layer behaviour. The experiments showed why ReLU is often preferred for hidden layers in modern neural networks.